# reflection-method: Finding Eclipsing Binary Minima

This notebook demonstrates how to use the `reflection-method` library to find the minimum of an eclipsing binary's light curve using the reflection method.

## Installation

```bash
pip install "reflection-method[plot]"
```

In [ ]:
import numpy as np
from reflection_method import find_minimum, find_x0, fit_spline, combine
from scipy.interpolate import UnivariateSpline
import matplotlib.pyplot as plt
from reflection_method.plot import plot_all

%matplotlib inline

## 1. Generate Synthetic Data (or load your data)

In [ ]:
rng = np.random.default_rng(42)
n = 400
phase = np.sort(rng.uniform(0.0, 1.0, n))

# Primary minimum at phase 0.5
primary = 0.7 * np.exp(-((phase - 0.5) ** 2) / (2 * 0.02**2))
# Secondary minima
secondary = 0.15 * (
    np.exp(-((phase - 0.0) ** 2) / (2 * 0.03**2))
    + np.exp(-((phase - 1.0) ** 2) / (2 * 0.03**2))
)
flux = 1.0 - primary - secondary + rng.normal(0.0, 0.01, n)

# Quick look at the data
plt.figure(figsize=(8, 4))
plt.scatter(phase, flux, s=8, alpha=0.6)
plt.xlabel('Phase')
plt.ylabel('Relative Flux')
plt.title('Synthetic Eclipsing Light Curve')
plt.grid(True, alpha=0.3)
plt.show()

## 2. One-Line Full Pipeline

In [ ]:
from reflection_method import find_minimum

result = find_minimum(
    phase, flux,
    pts_per_knot=10,
    degree=3,
    n_scan=200,
    n_bootstrap=60,
    n_scan_boot=80,
    rng=np.random.default_rng(123)
)

print(f"Minimum at x0 = {result.x0:.4f} ± {result.x0_std:.4f}")
print(f"68% CI: [{result.x0_lo:.4f}, {result.x0_hi:.4f}]")
print(f"Sigma min: {result.sigma_min:.4f}")
print(f"N points: {result.n_points}, Bootstrap iterations: {result.n_bootstrap}")

## 3. Step-by-Step (for inspection/debugging)

In [ ]:
# 3a. Initial spline fit
spl1 = fit_spline(phase, flux, pts_per_knot=10, degree=3)

# 3b. Scan x0 to find minimum of σ₂
x0_opt, x0_grid, sigma2 = find_x0(
    phase, flux,
    pts_per_knot=10,
    degree=3,
    n_scan=200
)
print(f"x0_opt = {x0_opt:.4f}")

# 3c. Spline through σ₂ for refinement
spl_sigma = UnivariateSpline(x0_grid, sigma2, k=3, s=0)

# 3d. Bootstrap uncertainty
from reflection_method import bootstrap_x0
x0_std, x0_lo, x0_hi = bootstrap_x0(
    phase, flux, x0_opt, spl1,
    pts_per_knot=10, degree=3, w=None,
    n_bootstrap=60, n_scan_boot=80,
    rng=np.random.default_rng(123)
)
print(f"x0_std = {x0_std:.4f}, CI = [{x0_lo:.4f}, {x0_hi:.4f}]")

# 3e. Reflected spline for plotting
xr = 2 * x0_opt - phase
x_all, y_all, _ = combine(phase, flux, x0_opt)
spl2 = fit_spline(x_all, y_all, 20, 3)

# 3f. Bootstrap samples for histogram
from reflection_method.core import spline_variance, bootstrap_x0 as _bx0
from scipy.optimize import minimize_scalar
rng2 = np.random.default_rng(123)
resid = flux - spl1(phase)
n_pts = len(phase)
x0_boot = np.empty(60)
for k in range(60):
    y_boot = spl1(phase) + resid[rng2.integers(0, n_pts, n_pts)]
    spl_b = fit_spline(phase, y_boot, 10, 3)
    x_fine = np.linspace(float(phase.min()), float(phase.max()), 1001)
    center = float(x_fine[int(np.argmin(spl_b(x_fine)))])
    win = 0.1 * (float(phase.max()) - float(phase.min()))
    lo = max(float(phase.min()), center - win / 2)
    hi = min(float(phase.max()), center + win / 2)
    grid = np.linspace(lo, hi, 80)
    sig = np.empty(len(grid))
    for i, x0 in enumerate(grid):
        xs, ys, _ = combine(phase, y_boot, x0)
        sp = fit_spline(xs, ys, 20, 3)
        sig[i] = np.sqrt(spline_variance(sp, xs, ys))
    sig = np.maximum(sig, 1e-6)
    spl_sig = UnivariateSpline(grid, sig, k=3, s=0)
    x0_boot[k] = minimize_scalar(spl_sig, bounds=(grid[0], grid[-1]), method='bounded').x

## 4. Full 4-Panel Plot

In [ ]:
from reflection_method.plot import plot_all

fig = plot_all(
    phase, flux, spl1, xr, spl2,
    x0_opt, x0_std,
    x0_grid, sigma2, spl_sigma, x0_boot,
    xlabel='Phase', ylabel='Relative Flux', x_unit=''
)
plt.show()

# Save to file
# fig.savefig('eclipsing_minimum.png', dpi=150, bbox_inches='tight')

## 5. Using Real AAVSO Data (via CLI)

```bash
# Install with CLI support
pip install "reflection-method[cli,plot]"

# Run on AAVSO CSV file
reflection-method find data.csv \
    -x DATE-OBS -y MAG -w MAG_ERR \
    -t iso -k 10 -d 3 -n 200 -b 60 -s 42 \
    -p output.png
```

The CLI handles:
- AAVSO extended format (comments, header in `#NAME,DATE-OBS,...` line)
- Time formats: `iso` (DATE-OBS→UTC), `jd`, `hjd`, `mjd`, `phase`, `minutes`
- Magnitudes used directly on the logarithmic scale (`find_peak=True` internally)
- Weights from MAG_ERR
- JSON output with x0, uncertainties, UTC time, and parameters
- Optional 4-panel plot (`--plot output.png`)